## Making a free Account on SerpAPI to get Flight Prices:

1. Go to SerpApi.com and create a free account.

2. Go to your dashboard and copy your API Key.

In [4]:
# Testing SerpAPI Connection:

import os
import requests
from dotenv import load_dotenv
load_dotenv(override= True)
flight_api_key = os.getenv('FLIGHT_API_KEY')

def get_live_price(origin_iata, destination_iata, outbound_date):

    print(f'Fetching live prices for {origin_iata} to {destination_iata}...')

    url = 'https://serpapi.com/search'

    params = {
        'engine': 'google_flights',
        'departure_id': origin_iata,
        'arrival_id': destination_iata,
        'outbound_date': outbound_date, #YYYY-MM-DD
        'type': 2,
        'currency': 'USD',
        'hl': 'en',
        'api_key': flight_api_key
    }

    response = requests.get(url, params=params)
    data = response.json()
    #print(data)

    # Extracting just the price of first 'best flight':
    try:
        best_flight = data['best_flights'][0]
        price = best_flight['price']
        airline = best_flight['flights'][0]['airline']
        return f'The cheapest live flight is ${price} on {airline}.'

    except KeyError:
        return 'Could not find flight data. Check your dates or IATA codes.'

# Testing:
print(get_live_price('JFK', 'LHR', '2026-03-30'))

Fetching live prices for JFK to LHR...
The cheapest live flight is $395 on British Airways.


## Chat Interface:

In [7]:
# Imports:

import gradio as gr
import os
from dotenv import load_dotenv
from openai import OpenAI
from datetime import datetime

In [8]:
# System Prompt:

today_date = datetime.now()

system_prompt= f'''
You are a helpful assistant for an Airline called FlightAI.
Today's date is {today_date}.
Give short, courteous answers.
If a user asks for a flight, use your tool to check live prices.
Always convert city names to 3-letter IATA airport codes before calling the tool.
'''

In [15]:
# Function to Get Ticket Price:
def get_live_price(origin_iata, destination_iata, outbound_date):

    print(f'Fetching live prices for {origin_iata} to {destination_iata}...')

    url = 'https://serpapi.com/search'

    params = {
        'engine': 'google_flights',
        'departure_id': origin_iata,
        'arrival_id': destination_iata,
        'outbound_date': outbound_date, #YYYY-MM-DD
        'type': 2,
        'currency': 'INR',
        'hl': 'en',
        'api_key': flight_api_key
    }

    response = requests.get(url, params=params)
    data = response.json()
    #print(data)

    # Extracting just the price of first 'best flight':
    try:
        best_flight = data['best_flights'][0]
        price = best_flight['price']
        airline = best_flight['flights'][0]['airline']
        return f'The cheapest live flight is {price} INR on {airline}.'

    except KeyError:
        return 'Could not find flight data. Check your dates or IATA codes.'

In [16]:
get_live_price('BOM', 'LHR', '2026-03-30')

Fetching live prices for BOM to LHR...


'The cheapest live flight is 130241 INR on Oman Air.'

In [17]:
# Defining JSON Function Descriptions for LLM:

get_ticket_price_schema = {
    'name': 'get_ticket_price',
    'description': 'Get the REAL-TIME live ticket price for a one-way flight.',
    'parameters': {
        'type': 'object',
        'properties': {
            'origin_iata': {'type': 'string',
                            'description': 'The 3-letter IATA airport code for the origin city.'},
            'destination_iata': {'type': 'string',
                                 'description': 'The 3-letter IATA airport code for the destination city.'},
            'outbound_date': {'type': 'string',
                              'description': 'The date of the flight in exact YYYY-MM-DD format.'}
        },
        'required': ['origin_iata', 'destination_iata', 'outbound_date'],
        'additionalProperties': False
    }
}

In [18]:
# Adding function to main Tool List:
tools = [
    {'type': 'function', 'function': get_ticket_price_schema}
]